<a href="https://colab.research.google.com/github/GilbertOwen/reefscan-id/blob/noaa-check/notebooks/03_eda_noaa.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ReefScan-ID: NOAA bleaching dataset check

Check of the NOAA PIFSC ESD Coral Bleaching dataset before it is used for stage 2. Run in Google Colab; no GPU is needed.

### Setup: libraries

In [3]:
import huggingface_hub
import pandas as pd

print("huggingface_hub version: {}".format(huggingface_hub.__version__))
print("pandas version: {}".format(pd.__version__))

huggingface_hub version: 1.33.0
pandas version: 2.2.3


### Setup: Google Drive

In [4]:
import os
from google.colab import drive

drive.mount("/content/drive")

base_dir = "/content/drive/MyDrive/ReefScan-ID"
noaa_dir = base_dir + "/RawData/NOAA-PIFSC-ESD-CORAL-Bleaching-Dataset"
print("NOAA folder found: {}".format(os.path.isdir(noaa_dir)))
print("Contents: {}".format(sorted(os.listdir(noaa_dir))))

Mounted at /content/drive
NOAA folder found: True
Contents: ['.git', '.gitattributes', '.gitignore', 'README.md', 'test', 'train', 'val']


## Section 1: Do the labels load?

Question: does every image in the Drive copy of the dataset get one of the two labels (CORAL and CORAL_BL) from its folder name, with the counts the folders on the Hub hold?

Checks: count the image files in each split and class folder of the repository on the Hub, list the images in the Drive folder with the split and label taken from the path, then compare the counts per folder and print the label counts per split.

### Files in the repository

In [5]:
from collections import Counter
from huggingface_hub import list_repo_files

repo_id = "NMFS-OSI/NOAA-PIFSC-ESD-CORAL-Bleaching-Dataset"
image_endings = (".jpg", ".jpeg", ".png")

files = list_repo_files(repo_id, repo_type="dataset")
image_files = [name for name in files if name.lower().endswith(image_endings)]
other_files = [name for name in files if not name.lower().endswith(image_endings)]

print("Files in the repository: {}".format(len(files)))
print("Image files: {}".format(len(image_files)))
print("Other files: {}".format(other_files[:20]))

folder_counts = Counter()
for name in image_files:
    folder_counts["/".join(name.split("/")[:-1])] += 1

for folder in sorted(folder_counts):
    print("{}: {} images".format(folder, folder_counts[folder]))

Files in the repository: 10422
Image files: 10419
Other files: ['.gitattributes', '.gitignore', 'README.md']
test/CORAL: 974 images
test/CORAL_BL: 591 images
train/CORAL: 4541 images
train/CORAL_BL: 2751 images
val/CORAL: 973 images
val/CORAL_BL: 589 images


### Images in the Drive folder

In [6]:
import time

start = time.time()
rows = []
skipped = []
for root, dirs, names in os.walk(noaa_dir):
    if ".git" in dirs:
        dirs.remove(".git")
    for name in names:
        relative = os.path.relpath(os.path.join(root, name), noaa_dir).replace(os.sep, "/")
        parts = relative.split("/")
        if name.lower().endswith(image_endings) and len(parts) == 3:
            rows.append({"split": parts[0], "label": parts[1], "path": relative})
        else:
            skipped.append(relative)

noaa = pd.DataFrame(rows)
print("Listing took {:.1f} seconds".format(time.time() - start))
print("Images found: {}".format(len(noaa)))
print("Files skipped: {}, first ones: {}".format(len(skipped), skipped[:20]))
print("Splits: {}".format(sorted(noaa["split"].unique())))
print("Labels: {}".format(sorted(noaa["label"].unique())))
print(noaa.head(5))

Listing took 26.9 seconds
Images found: 10419
Files skipped: 3, first ones: ['.gitattributes', '.gitignore', 'README.md']
Splits: ['test', 'train', 'val']
Labels: ['CORAL', 'CORAL_BL']
  split     label                                      path
0   val  CORAL_BL  val/CORAL_BL/LAN-B1820_2019_13_10127.PNG
1   val  CORAL_BL  val/CORAL_BL/LAN-B1824_2019_05_10411.PNG
2   val  CORAL_BL   val/CORAL_BL/LAN-B1800_2019_19_9263.PNG
3   val  CORAL_BL   val/CORAL_BL/LAN-B1808_2019_08_9776.PNG
4   val  CORAL_BL    val/CORAL_BL/KUR-B016_2019_10_6567.PNG


### Row counts against the folders

In [7]:
drive_counts = Counter(noaa["split"] + "/" + noaa["label"])

for folder in sorted(set(folder_counts) | set(drive_counts)):
    print("{}: {} on the Hub, {} on Drive, same: {}".format(
        folder, folder_counts[folder], drive_counts[folder], folder_counts[folder] == drive_counts[folder]))
print("Total: {} on the Hub, {} on Drive".format(sum(folder_counts.values()), len(noaa)))

print(pd.crosstab(noaa["split"], noaa["label"]))

test/CORAL: 974 on the Hub, 974 on Drive, same: True
test/CORAL_BL: 591 on the Hub, 591 on Drive, same: True
train/CORAL: 4541 on the Hub, 4541 on Drive, same: True
train/CORAL_BL: 2751 on the Hub, 2751 on Drive, same: True
val/CORAL: 973 on the Hub, 973 on Drive, same: True
val/CORAL_BL: 589 on the Hub, 589 on Drive, same: True
Total: 10419 on the Hub, 10419 on Drive
label  CORAL  CORAL_BL
split                 
test     974       591
train   4541      2751
val      973       589


### Paths that differ between the Hub and Drive

In [8]:
import re

hub_paths = set(image_files)
drive_paths = set(noaa["path"])

only_on_drive = sorted(drive_paths - hub_paths)
only_on_hub = sorted(hub_paths - drive_paths)

print("On Drive but not on the Hub: {}".format(len(only_on_drive)))
for path in only_on_drive[:50]:
    print("  {}".format(path))

print("On the Hub but not on Drive: {}".format(len(only_on_hub)))
for path in only_on_hub[:50]:
    print("  {}".format(path))

# A copy suffix such as " (1)" before the file ending is removed first
base_names = {}
for path in sorted(drive_paths):
    name = path.split("/")[-1]
    base = re.sub(r" \(\d+\)(\.\w+)$", r"\1", name)
    base_names.setdefault(base, []).append(path)

repeated = {base: paths for base, paths in base_names.items() if len(paths) > 1}
print("File names that appear more than once on Drive: {}".format(len(repeated)))
for base in sorted(repeated)[:50]:
    print("  {}: {}".format(base, repeated[base]))

duplicate_paths = noaa[noaa["path"].duplicated(keep=False)]
print("Rows with the same path listed more than once: {}".format(len(duplicate_paths)))
print(duplicate_paths.head(10))

On Drive but not on the Hub: 0
On the Hub but not on Drive: 0
File names that appear more than once on Drive: 0
Rows with the same path listed more than once: 0
Empty DataFrame
Columns: [split, label, path]
Index: []


**Observation**: All 10,419 images in Google Drive match the Hugging Face repository counts across all six split and class folders. No missing files, extra files, or duplicate paths were found, confirming that the dataset copy is consistent and images can be labelled using their folder names (CORAL and CORAL_BL).

## Section 2: NOAA file name groups

Question: do the first two parts of a NOAA file name identify a site, does the third part identify an original photo, and how many validation patches share a group with a train patch?

Checks: split every train and val file name into its parts, count what the parts look like, test whether a group of the first three parts behaves like one photo, then compare train and val for each candidate grouping, compare the labels inside groups, and compare the overlap with what a random assignment of the same split sizes gives. Only the train and val folders are used, the test folder is left out, and no image is opened.

### Parts of the file names

In [ ]:
import numpy as np

measured = noaa[noaa["split"].isin(["train", "val"])].copy()
print("Names measured: {} (train {}, val {}); the test folder is left out".format(
    len(measured), (measured["split"] == "train").sum(), (measured["split"] == "val").sum()))

stems = measured["path"].str.split("/").str[-1].str.rsplit(".", n=1).str[0]
stem_parts = stems.str.split("_")
print("Number of parts per name: {}".format(stem_parts.str.len().value_counts().to_dict()))

measured = measured[stem_parts.str.len() == 4].copy()
stem_parts = stem_parts[stem_parts.str.len() == 4]
measured["site"] = stem_parts.str[0]
measured["year"] = stem_parts.str[1]
measured["third"] = pd.to_numeric(stem_parts.str[2], errors="coerce")
measured["fourth"] = pd.to_numeric(stem_parts.str[3], errors="coerce")
measured["island"] = measured["site"].str.split("-").str[0]
measured["first_two"] = measured["site"] + "_" + measured["year"]
measured["first_three"] = measured["first_two"] + "_" + stem_parts.str[2]

print("Names with four parts: {}".format(len(measured)))
print("Third or fourth part that is not a number: {}".format(measured[["third", "fourth"]].isna().any(axis=1).sum()))
print("First part looks like three letters, a dash and a code: {:.1f}%".format(
    100 * measured["site"].str.match(r"^[A-Za-z]{3}-[A-Za-z]*[0-9]+$").mean()))
print("Years: {}".format(measured["year"].value_counts().sort_index().to_dict()))
print("Three-letter codes before the dash: {}".format(measured["island"].value_counts().to_dict()))
print("Third part: smallest {}, largest {}".format(measured["third"].min(), measured["third"].max()))
print("Fourth part: smallest {}, largest {}, distinct values {} for {} names".format(
    measured["fourth"].min(), measured["fourth"].max(), measured["fourth"].nunique(), len(measured)))

### Is the first part a site?

In [ ]:
sites = measured.groupby("site").agg(patches=("path", "size"), years=("year", "nunique"))
print("First parts (candidate sites): {}".format(len(sites)))
print("Candidate sites seen in more than one year: {}".format((sites["years"] > 1).sum()))

site_years = measured.groupby("first_two").agg(
    patches=("path", "size"), third_values=("third", "nunique"), third_largest=("third", "max"))
print("Site-years: {}".format(len(site_years)))
print(site_years.describe(percentiles=[0.1, 0.5, 0.9]).round(1).to_string())

# InPort record 67962 gives 5 to 72 images per site
print("Site-years with more than 72 distinct third values: {}".format((site_years["third_values"] > 72).sum()))
print("Site-years whose largest third value is above 72: {}".format((site_years["third_largest"] > 72).sum()))

### Is the third part an original photo?

In [ ]:
group_sizes = measured.groupby("first_three").size()
print("Groups of the first three parts: {}".format(len(group_sizes)))
print("Patches per group: {}".format(group_sizes.describe(percentiles=[0.5, 0.9, 0.99]).round(1).to_dict()))
print("Largest groups (patches): {}".format(group_sizes.sort_values(ascending=False).head(5).to_dict()))

# A 2 by 5 grid gives at most 10 patches per photo (dataset card, as noted earlier)
print("Groups with more than 10 patches: {}".format((group_sizes > 10).sum()))

spans = measured.groupby("first_three")["fourth"].agg(lambda values: values.max() - values.min() + 1)
print("Span of the fourth part inside a group (largest minus smallest plus 1): {}".format(
    spans.describe(percentiles=[0.5, 0.9, 0.99]).round(1).to_dict()))
print("Groups whose span is 10 or less: {} of {}".format((spans <= 10).sum(), len(spans)))

ordered = measured.sort_values("fourth")
consecutive = ordered["fourth"].diff() == 1
print("Pairs of patches with consecutive fourth parts: {}".format(consecutive.sum()))
for column in ["first_three", "first_two", "site", "island"]:
    same = ordered[column] == ordered[column].shift()
    print("  same {}: {:.1f}%".format(column, 100 * (same & consecutive).sum() / consecutive.sum()))

chance = (group_sizes ** 2).sum() / len(measured) ** 2
print("Chance that two random patches share a first-three group, approximately: {:.2f}%".format(100 * chance))

### Groups shared between train and val

In [ ]:
groupings = {"first part": "site", "first two parts": "first_two", "first three parts": "first_three"}
val_rows = measured[measured["split"] == "val"]

for name, column in groupings.items():
    splits_per_group = measured.groupby(column)["split"].nunique()
    shared = splits_per_group[splits_per_group > 1].index
    in_shared = val_rows[column].isin(shared).sum()
    print("{}: {} groups, {} in both train and val, {} of {} val patches in a shared group".format(
        name, len(splits_per_group), len(shared), in_shared, len(val_rows)))
    for group in list(shared[:3]):
        rows = measured[measured[column] == group]
        examples = list(zip(rows["split"], rows["path"].str.split("/").str[-1]))[:4]
        print("  example group {}: {}".format(group, examples))

### Labels inside groups

In [ ]:
train_rows = measured[measured["split"] == "train"]

for name, column in groupings.items():
    labels_per_group = measured.groupby(column)["label"].nunique()
    print("{}: groups with both labels: {} of {}".format(name, (labels_per_group > 1).sum(), len(labels_per_group)))

    train_majority = train_rows.groupby(column)["label"].agg(lambda values: values.value_counts().idxmax())
    known = val_rows[val_rows[column].isin(train_majority.index)]
    agree = (known["label"] == known[column].map(train_majority)).mean()
    print("{}: {} val patches have a train group, label equals the train majority of its group: {:.1f}%".format(
        name, len(known), 100 * agree))

print("Always answering CORAL on val is right for {:.1f}% of val patches".format(
    100 * (val_rows["label"] == "CORAL").mean()))

### Same split sizes assigned at random

In [ ]:
shuffles = 20
shuffle_seed = 0
rng = np.random.default_rng(shuffle_seed)
is_val = (measured["split"] == "val").to_numpy()
print("Seed: {}, shuffles: {}; the split sizes are kept, the label balance is not".format(shuffle_seed, shuffles))

for name, column in groupings.items():
    codes = pd.factorize(measured[column])[0]
    group_count = codes.max() + 1

    def overlap(flags):
        in_val = np.bincount(codes[flags], minlength=group_count) > 0
        in_train = np.bincount(codes[~flags], minlength=group_count) > 0
        shared = in_val & in_train
        return shared.sum(), shared[codes[flags]].sum()

    observed = overlap(is_val)
    random_runs = np.array([overlap(rng.permutation(is_val)) for _ in range(shuffles)])
    print("{}: observed {} shared groups and {} val patches in shared groups".format(name, observed[0], observed[1]))
    print("{}: random assignment gives {} to {} shared groups (mean {:.0f}) and {} to {} val patches (mean {:.0f})".format(
        name, random_runs[:, 0].min(), random_runs[:, 0].max(), random_runs[:, 0].mean(),
        random_runs[:, 1].min(), random_runs[:, 1].max(), random_runs[:, 1].mean()))

**Observation**: